In [1]:
import math
import os
import pandas as pd
import numpy as np


In [2]:
#calculate passage number when I ran out of passage 5 vials ;;
def passage_number(time):
    base_vial = 6
    if time > 2:
        base_vial = base_vial-1 
    pnum = base_vial + (3 * time)
    return pnum



def well_namer(row, col):
    well_name = str(chr(ord('@')+ row)) + str(col).rjust(2, '0')  #make the number have a left align, adding a zero
    return well_name

import re

def extract_number(string):
    match = re.search(r'\d+', string)
    if match:
        return int(match.group())
    else:
        return None




In [3]:
#Get 96-well plate CSV file from benchiling/Notion NOTE: don't initialze an empty df, use a list and convert after.

plate_path = 'plate_metadata/' #'/home/mattiazzilab/Documents/Allie_Scripts/May 8 seeding.csv'

export_path = 'plate_metadata/'#'/mnt/bigdisk1/Allie_S/Replicative_Age_Project/Data Mining/metadata/'

def load_plate_df(path):    
    raw_plate_df = pd.read_csv(path, header=0, usecols=range(1,13)).dropna(axis=1,how='all')
    plate_df = raw_plate_df.dropna(axis=0, thresh=2)
    print(plate_df.shape) #Checks if correct number of rows and columns
    
    return plate_df

def make_map_df():
    '''
    Return a df with the required columns
    '''
    plate_map_df = pd.DataFrame(columns = ['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field',
                                'TimepointName', 'SerialPassage_BatchNumber', 'AgeGroup', 'PassageNumber', 'Staining', 'Drug'])
    print(plate_map_df.columns)
    return plate_map_df
#plate_df.head(13)
#print(columns_row)

In [4]:
def export_platemap_csv(plate_df, export_path):
    plate_map_df = make_map_df()
    columns_row = plate_df.columns#get_columns_row(plate_df)
    for index,data in plate_df.iterrows():
        row = data.to_list()
        for count,value in enumerate(row):
            curr_well = value
            if pd.isna(curr_well):
                continue
            row_list = []
            # get string data and label of row in df (e.g. col1, text= R1T0 EAA1-488 Tfn-647)
            # regex to separate into different variables
            # then add them to dict with their respective col index (label) and index of the row in the column (column.index)
            for i in range(40):
                
                row_entry = {}
                
                row_index = index+1  #Make it 1-indexed
                
                column_index = columns_row[count]  # Use header row for column index
                #print(column_index)
            
                if ~np.isnan(row_index): 
                    well_name = well_namer(row_index, column_index)
                else:
                    well_name = 'Empty'
                    continue
                
                #seperate well metadata by space
                text = curr_well.split(' ')
                
                #use regex to extract the numerical bits
                serial_passage_batch = extract_number(text[0])
                age_group = extract_number(text[1]) #time used to mean age group - deprecated term but still used in code
                passage_num = extract_number(text[2]) #passage_number(Int(time)) - use the function if you don;t have passage number in the table
                
                #grab drug, name and stains
                if '_' in text[1]: #if there is an underscore than the well is drug-treated for this group
                    drug = text[1].split('_')[1] #grab the drug name past the underscore
                else:
                    drug = 'None'
                
                name = text[0] + ' ' + text[1] + ' ' + text[2]
                stains = ' '.join(text[3:len(text)]) #Use if 3 passage number is in the table values
                
                field = i+1 #information for the field is just 1-40, nothing else changes

                print(name, serial_passage_batch, age_group, passage_num, field, stains, drug, sep=',') 
        
                row_entry.update({'Metadata_Well': well_name, 'Metadata_WellRow': row_index, 'Metadata_WellColumn': column_index, 'Metadata_Field': field,
                                'TimepointName':name, 'SerialPassage_BatchNumber': serial_passage_batch, 'AgeGroup': age_group, 'PassageNumber': passage_num, 'Staining': stains, 'Drug': drug })

                row_list.append(row_entry)
                
            rows = pd.DataFrame(row_list)
            plate_map_df = pd.concat([plate_map_df, rows],ignore_index=True)
            #column_index = column_index+1
    plate_map_df.to_csv(os.path.join(export_path,'map.csv'), index=False)
    

In [5]:
for root, dirs, files in os.walk(plate_path):
    for filename in files:
        if filename.endswith(".csv") and "map" not in filename:
            file_path = os.path.join(root,filename)
            plate_df = load_plate_df(os.path.abspath(file_path))
            display(plate_df)
            export_path = os.path.abspath(root)
            print(f"Exporting {filename} to {export_path}")
            export_platemap_csv(plate_df, export_path)
            

(6, 12)


,1,2,3,4,5,6,7,8,9,10,11,12
1,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG2 P14 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:800 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:800 + Phalloi...
2,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG2 P14 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:1600 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:1600 + Phallo...
3,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG2 P14 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:2600 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:2600 + Phallo...
4,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG2 P14 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:3600 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:3600 + Phallo...
5,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG2 P14 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:3600 LMNB1 1:400 +...,SPB5 AG3_Doxo P16 p16INK4A-594 1:3600 LMNB1 1:...
6,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG2 P14 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p21-594 1:6000 + Phalloidin-647,SPB5 AG3_Doxo P16 p21-594 1:600 + Phalloidin-647


Exporting mar28plate.csv to /Users/allielas/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'TimepointName', 'SerialPassage_BatchNumber',
       'AgeGroup', 'PassageNumber', 'Staining', 'Drug'],
      dtype='object')
SPB8 AG3_Doxo P16,8,3,16,1,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,2,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,3,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,4,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,5,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,6,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,7,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,8,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,9,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,10,LAMP1-488 + MitoRed + Phal

,2,3,4,5,6,7,8,9,10,11
1,SPB2 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG2 P12 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB6 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG3 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG2 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG1 EEA1-488 + ConA-594 + Tfn-647,SPB6 AG0 EEA1-488 + ConA-594 + Tfn-647
2,SPB2 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG2 P12 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB6 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG3 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG2 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG1 EEA1-488 + ConA-594 + Tfn-647,SPB6 AG0 EEA1-488 + ConA-594 + Tfn-647
3,SPB2 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG2 P12 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB6 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG3 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG2 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG1 EEA1-488 + ConA-594 + Tfn-647,SPB6 AG0 EEA1-488 + ConA-594 + Tfn-647
4,SPB2 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG2 P12 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB6 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG4 P17 EEA1-488 + ConA-594 + Tfn-647 + IKA,SPB3 AG3 EEA1-488 + ConA-594 + Tfn-647+ IKA,SPB4 AG2 EEA1-488 + ConA-594 + Tfn-647+ IKA,SPB5 AG1 EEA1-488 + ConA-594 + Tfn-647+ IKA,SPB6 AG0 EEA1-488 + ConA-594 + Tfn-647+ IKA
5,SPB2 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG2 P12 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB6 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG4 P17 EEA1-488 + Tfn-568 + WGA-CF640R,SPB3 AG3 EEA1-488 + Tfn-568 + WGA-CF640R,SPB4 AG2 EEA1-488 + Tfn-568 + WGA-CF640R,SPB5 AG1 EEA1-488 + Tfn-568 + WGA-CF640R,SPB6 AG0 EEA1-488 + Tfn-568 + WGA-CF640R
6,SPB1 AG5 P20 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG5 P20 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG5 P20 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG5 P20 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG5 P20 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG5 P20 EEA1-488 + ConA-594 + Tfn-647,SPB1 AG5 P20 EEA1-488 + ConA-594 + Tfn-647,SPB1 AG5 P20 EEA1-488 + ConA-594 + Tfn-647,SPB1 AG5 P20 EEA1-488 + ConA-594 + Tfn-647,SPB1 AG5 P20 EEA1-488 + ConA-594 + Tfn-647


Exporting mar26plate.csv to /Users/allielas/HTP_fibroblasts_mitolyso/plate_metadata/20240326_rep02_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'TimepointName', 'SerialPassage_BatchNumber',
       'AgeGroup', 'PassageNumber', 'Staining', 'Drug'],
      dtype='object')
SPB2 AG4 P17,2,4,17,1,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,2,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,3,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,4,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,5,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,6,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,7,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,8,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,9,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,10,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,2,4,17,11,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB2 AG4 P17,

...

Exporting apr10plate.csv to /Users/allielas/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'TimepointName', 'SerialPassage_BatchNumber',
       'AgeGroup', 'PassageNumber', 'Staining', 'Drug'],
      dtype='object')
SPB8 AG3_Doxo P16,8,3,16,1,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,2,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,3,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,4,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,5,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,6,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,7,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,8,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,9,LAMP1-488 + MitoRed + Phalloidin-647,Doxo
SPB8 AG3_Doxo P16,8,3,16,10,LAMP1-488 + MitoRed + Phal

,2,3,4,5,6,7,8,9,10,11
1,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
2,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
3,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
4,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
5,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
6,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647


Exporting mar13plate.csv to /Users/allielas/HTP_fibroblasts_mitolyso/plate_metadata/20240313_rep01_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'TimepointName', 'SerialPassage_BatchNumber',
       'AgeGroup', 'PassageNumber', 'Staining', 'Drug'],
      dtype='object')
SPB1 AG4 P17,1,4,17,1,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,2,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,3,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,4,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,5,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,6,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,7,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,8,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,9,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,10,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,1,4,17,11,LAMP1-488 + MitoRed + WGA-CF640R,None
SPB1 AG4 P17,

,2,3,4,5,6,7
0,SPB2 AG5 P21 Phalloidin-647 + ConA-594,SPB3 AG4 P18 Phalloidin-647 + ConA-594,SPB4 AG3 P15 Phalloidin-647 + ConA-594,SPB5 AG2 P13 Phalloidin-647 + ConA-594,SPB6 AG1 P10 Phalloidin-647 + ConA-594,SPB7 AG0 P8 Phalloidin-647 + ConA-594
1,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
2,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
3,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
4,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
5,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
6,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
7,SPB2 AG5 P21 Phalloidin-647 + WGA-647 + ConA-594,SPB3 AG4 P18 Phalloidin-647 + WGA-647 + ConA-594,SPB4 AG3 P15 Phalloidin-647 + WGA-647 + ConA-594,SPB5 AG2 P13 Phalloidin-647 + WGA-647 + ConA-594,SPB6 AG1 P10 Phalloidin-647 + WGA-647 + ConA-594,SPB7 AG0 P8 Phalloidin-647 + WGA-647 + ConA-594


Exporting oct18plate.csv to /Users/allielas/HTP_fibroblasts_mitolyso/plate_metadata/20241018_rep03_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'TimepointName', 'SerialPassage_BatchNumber',
       'AgeGroup', 'PassageNumber', 'Staining', 'Drug'],
      dtype='object')
SPB2 AG5 P21,2,5,21,1,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,2,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,3,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,4,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,5,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,6,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,7,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,8,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,9,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,10,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,11,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,12,Phalloidin-647 + ConA-594 ,None
SPB2 AG5 P21,2,5,21,13,P